=============================================================================
MISSOURI VOTER RESOURCE ALLOCATION PROJECT
=============================================================================
Course: CAPS 5576 - Analytics Applications
Team: Group 1
Project: Prescriptive and Predictive Voter Resource Allocation
Focus: Missouri Presidential Elections (2016, 2020, 2024)
=============================================================================

COLLABORATION NOTE:
-------------------
This notebook was initially built in an individual environment (SNOWBEARAIR_DB).
Once it was validated, it was ported here to a shared team GitHub account
where all 4 team members can collaborate.

=============================================================================

FILES REQUIRED (upload as notebook assets before execution):
------------------------------------------------------------

ELECTION DATA (3 files):
- MO 2016 Election Results.csv
- MO 2020 Election Results.csv
- MO 2024 Election Results.csv

CENSUS DATA - 2016 (5 files):
- MO 2016 Census Income.csv
- MO 2016 Census Education.csv
- MO 2016 Census Race.csv
- MO 2016 Census Commute.csv
- MO 2016 Census Sex by Age.csv

CENSUS DATA - 2020 (5 files):
- MO 2020 Census Income.csv
- MO 2020 Census Education.csv
- MO 2020 Census Race.csv
- MO 2020 Census Commute.csv
- MO 2020 Census Sex by Age.csv

CENSUS DATA - 2024 (5 files):
- MO 2024 Census Income.csv
- MO 2024 Census Education.csv
- MO 2024 Census Race.csv
- MO 2024 Census Commute.csv
- MO 2024 Census Sex by Age.csv

POLLING LOCATION DATA (1 file):
- MO 2020 Polling Locations.csv

SHAPEFILE DATA (5 files):
- MO 2020 Precincts.shp
- MO 2020 Precincts.dbf
- MO 2020 Precincts.shx
- MO 2020 Precincts.prj
- MO 2020 Precincts.cpg

TOTAL: 24 files

=============================================================================

NOTEBOOK STRUCTURE:
-------------------
SECTION 1: Documentation & Setup

SECTION 2: Data Ingestion

SECTION 3: Data Cleaning - Election Data

SECTION 4: Data Cleaning - Census Data

SECTION 5: Data Cleaning - Polling & Shapefile

SECTION 6: Write Staging Tables to Snowflake

SECTION 7: SQL Aggregations & JOINs

SECTION 8: Data Quality Validation

SECTION 9: Exploratory Data Analysis

SECTION 10: Geospatial Analysis - inserted for David's geospatial idea

SECTION 11: Summary & Next Steps

=============================================================================

# Missouri Voter Resource Allocation Project

## Project Goal
Analyze historical presidential election results alongside demographic and geographic data to evaluate how polling resources could potentially be allocated more effectively across Missouri precincts.

## Analysis Focus
The analysis focuses on **three presidential election cycles**: 2016, 2020, and 2024.

Presidential election years were chosen because they produce the **highest voter turnout** and the **most consistent statewide participation**. Focusing on presidential election cycles provides a clearer signal for modeling voter demand and analyzing polling resource allocation across precincts.

## Data Architecture
The project integrates several categories of data:
- **Presidential election results** (precinct-level) - 2016, 2020, 2024
- **Census demographic data** (county-level) - ACS 5-Year Estimates aligned to each election
- **Polling location data** - Physical polling places by precinct
- **Precinct boundaries** (optional) - Geographic shapefiles for spatial analysis

## Programming Paradigms
- **Imperative (Python):** Data loading, cleaning, transformations
- **Declarative (SQL):** Joins, aggregations, analytical queries

## Coding Standards
- snake_case naming with meaningful variable names
- One output per cell
- Explanations in Markdown cells (not inline comments)
- Pandas method chaining
- List comprehensions over loops

# Data Sources

## Election Results Data

**Source:** OpenElections Project (GitHub)  
https://github.com/openelections/openelections-data-mo

**Files:**
- MO 2016 Election Results.csv (128,859 rows)
- MO 2020 Election Results.csv (132,123 rows)
- MO 2024 Election Results.csv (190,270 rows)

These datasets contain precinct-level vote totals for each candidate and office. Each row represents the vote total for one candidate within a specific precinct.

**Schema Note:** The 2020 and 2024 files include a `precinct_code` column not present in 2016. This column is dropped during preprocessing to maintain a consistent schema.

## Census Demographic Data

**Source:** U.S. Census Bureau – American Community Survey (ACS) 5-Year Estimates  
https://data.census.gov

Demographic data was aligned with each presidential election year:
- **2016 Election** → ACS 2012-2016
- **2020 Election** → ACS 2016-2020
- **2024 Election** → ACS 2020-2024

**Tables Used (5 per year = 15 files total):**
- B01001 – Sex by Age
- B02001 – Race
- B08301 – Commuting / Transportation to Work
- B15003 – Educational Attainment
- B19013 – Median Household Income

All ACS datasets are at the **county level** for Missouri.

## Polling Location Data

**Source:** MIT Election Data and Science Lab  
https://electionlab.mit.edu/data

**File:** MO 2020 Polling Locations.csv (14,354 rows)

Contains polling locations associated with Missouri precincts. Although the data represents the 2020 election cycle, polling locations generally change slowly, making the dataset suitable for analysis across nearby election years.

## Precinct Geographic Files

**Source:** U.S. Census TIGER/Line Shapefiles  

**Files:** MO 2020 Precincts (.shp, .dbf, .shx, .prj, .cpg)

These files represent Missouri Voting Tabulation District (precinct) boundaries. They may be used for:
- Mapping turnout geographically
- Spatial analysis of polling locations
- Calculating distances between voters and polling places

# Data Preprocessing Requirements

## Election Data Preprocessing

1. **Add election year column** - Enables combining datasets from different cycles
2. **Remove precinct_code column** - Not present in 2016, dropped from 2020/2024
3. **Normalize precinct names** - Uppercase, remove special characters, trim whitespace
4. **Normalize county names** - Uppercase and trim whitespace
5. **Filter to Presidential results only** - Focus analysis on highest-turnout races

## Census Data Preprocessing

1. **Remove header artifact row** - ACS exports include a metadata row where GEO_ID = "Geography"
2. **Remove blank export columns** - Drop any "Unnamed" columns from export artifacts
3. **Extract county name** - Parse from NAME field, removing " County, Missouri" suffix
4. **Add census year column** - Track which ACS vintage the data comes from
5. **Convert to numeric types** - Ensure proper data types for analysis

## Data Preprocessing

### Election Data Preprocessing
Before combining election datasets, several preprocessing steps are performed.

Add election year column:

```python
df2016["year"] = 2016
df2020["year"] = 2020
df2024["year"] = 2024
```

Remove precinct_code where present:

```python
df = df.drop(columns=["precinct_code"], errors="ignore")
```

Normalize precinct names:

```python
df["precinct_clean"] = (
    df["precinct"]
    .astype(str)
    .str.upper()
    .str.replace("#","")
    .str.replace("  "," ")
    .str.strip()
)
```

### Census Data Preprocessing
Census datasets exported from data.census.gov contain some artifacts that should be cleaned.

Remove header artifact row:

```python
df = df[df["GEO_ID"] != "Geography"]
```

Remove blank export columns:

```python
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
```

Extract county name:

```python
df["county_clean"] = (
    df["NAME"]
    .str.replace(", Missouri", "", regex=False)
    .str.replace(" County", "", regex=False)
    .str.upper()
)
```

**Important – St. Louis County vs. St. Louis City:**
Missouri has both "St. Louis County" (FIPS 29189) and "St. Louis city" (FIPS 29510), which is an independent city that is not part of any county. The cleaning process preserves this distinction by only removing " County" from county names, not " city". This results in:

- St. Louis County → "ST. LOUIS"
- St. Louis city → "ST. LOUIS CITY"

**Warning:** If both entities are cleaned to the same name, JOINs between election and census data will produce duplicate rows (a cartesian product), leading to inflated row counts in analytical tables.

Add census year column:

```python
df["census_year"] = 2016  # or 2020 or 2024
```

## Snowflake Implementation Notes

### Column Naming and Case Sensitivity

Snowflake handles column names differently depending on how tables are created:

**Staging Tables (created via Python/Snowpark):**
- Column names are lowercase (e.g., `year`, `county_clean`, `votes`)
- Must use double quotes in SQL to reference them: `"year"`, `"county_clean"`

**Analytical Tables (created via SQL):**
- Use uppercase aliases when creating: `SELECT "year" AS YEAR, "county_clean" AS COUNTY`
- This allows unquoted references in downstream SQL: `WHERE YEAR = 2020`

### Snowflake Table Structure

**Staging Tables (loaded via Python):**

STG_ELECTION_RESULTS – Combined presidential election results (all years)

STG_CENSUS_INCOME – Median household income by county

STG_CENSUS_EDUCATION – Educational attainment by county

STG_CENSUS_RACE – Race demographics by county

STG_CENSUS_COMMUTE – Commuting patterns by county

STG_CENSUS_SEX_AGE – Sex and age demographics by county

STG_POLLING_LOCATIONS – Polling place locations

**Analytical Tables (created via SQL):**

PRECINCT_TURNOUT – Aggregated votes by precinct and year

COUNTY_TURNOUT – Aggregated votes by county and year

COUNTY_TURNOUT_TREND – Pivoted view with all years side by side

COUNTY_CENSUS – Combined census demographics with all years

COUNTY_ANALYSIS – Joined turnout and census data for analysis

COUNTY_POLLING_SUMMARY – Polling locations aggregated by county

COUNTY_VIZ_EXPORT – Final export table for visualization tools

### Expected Row Counts

| Table | Expected Rows | Notes |
|-------|---------------|-------|
| STG_ELECTION_RESULTS | ~70,000 | Presidential votes only |
| STG_CENSUS_* | 345 each | 115 counties × 3 years |
| STG_POLLING_LOCATIONS | ~14,000 | 2020 polling places |
| PRECINCT_TURNOUT | ~10,000 | Precincts × 3 years |
| COUNTY_TURNOUT | 348 | 116 counties × 3 years |
| COUNTY_TURNOUT_TREND | 117 | One row per county |
| COUNTY_CENSUS | 345 | 115 counties × 3 years |
| COUNTY_ANALYSIS | 117 | One row per county |
| COUNTY_POLLING_SUMMARY | 116 | One row per county |

Note: Missouri has 114 counties plus the independent city of St. Louis, for a total of 115 county-level entities. Some election data may show 116 counties due to Kansas City reporting.

## Verified Results

Statewide presidential election totals match official Missouri results:

| Year | Total Votes | Republican % | Democrat % |
|------|-------------|--------------|------------|
| 2016 | 2,808,298 | 56.78% | 38.14% |
| 2020 | 2,963,270 | 57.17% | 41.03% |
| 2024 | 2,995,327 | 58.49% | 40.08% |


In [1]:
# =============================================================================
# CONFIGURATION AND SETUP
# =============================================================================
# Update DATA_PATH to match your local folder structure.
# For GitHub: Use relative path 'Data/' if your folder structure is:
#   Final Project/
#   ├── Data/
#   │   ├── MO 2016 Election Results.csv
#   │   └── ... (all CSV and shapefile files)
#   └── MO_Voter_Project.ipynb

# ------------- UPDATE THIS PATH -------------
# DATA_PATH = 'Data/'  # Relative path for GitHub compatibility
DATA_PATH = '/Users/gas/Desktop/5576/Final Project/Data/'  # Or use absolute path
# --------------------------------------------

# =============================================================================
# LIBRARY IMPORTS
# =============================================================================
import pandas as pd
import numpy as np
import geopandas as gpd
import warnings
from getpass import getpass

warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
warnings.filterwarnings('ignore', category=UserWarning)

print("=" * 70)
print("LIBRARIES LOADED")
print("=" * 70)
print(f"pandas:    {pd.__version__}")
print(f"numpy:     {np.__version__}")
print(f"geopandas: {gpd.__version__}")
print(f"DATA_PATH: {DATA_PATH}")
print("=" * 70)


LIBRARIES LOADED
pandas:    2.2.3
numpy:     1.26.4
geopandas: 1.1.3
DATA_PATH: /Users/gas/Desktop/5576/Final Project/Data/


In [2]:
# =============================================================================
# SNOWFLAKE CONNECTION
# =============================================================================
# Each team member enters their own credentials when running the notebook.
# Credentials are NOT stored in the notebook file.

import snowflake.connector

print("Enter your Snowflake credentials:")
sf_user = input("Username: ")
sf_password = getpass("Password: ")

conn = snowflake.connector.connect(
    account='ytdayoz-nc71712',
    user=sf_user,
    password=sf_password,
    database='VOTER_PROJECT_DB',
    schema='ANALYTICS',
    warehouse='COMPUTE_WH'
)

cursor = conn.cursor()

print()
print("=" * 70)
print("CONNECTED TO SNOWFLAKE")
print("=" * 70)
print("Account:   ytdayoz-nc71712")
print("Database:  VOTER_PROJECT_DB")
print("Schema:    ANALYTICS")
print("Warehouse: COMPUTE_WH")
print("=" * 70)


Enter your Snowflake credentials:

CONNECTED TO SNOWFLAKE
Account:   ytdayoz-nc71712
Database:  VOTER_PROJECT_DB
Schema:    ANALYTICS
Warehouse: COMPUTE_WH


---
# SECTION 2: Data Ingestion
---

Load all raw data files into pandas DataFrames. Files are loaded exactly as downloaded to preserve raw source data for validation.


In [3]:
# Load Election Results (All Three Years)

raw_election_2016 = pd.read_csv(
    DATA_PATH + 'MO 2016 Election Results.csv', 
    keep_default_na=False, 
    na_values=['']
)
raw_election_2020 = pd.read_csv(
    DATA_PATH + 'MO 2020 Election Results.csv', 
    keep_default_na=False, 
    na_values=['']
)
raw_election_2024 = pd.read_csv(
    DATA_PATH + 'MO 2024 Election Results.csv', 
    keep_default_na=False, 
    na_values=['']
)

print("=" * 70)
print("ELECTION DATA LOADED")
print("=" * 70)
print(f"2016: {raw_election_2016.shape[0]:,} rows, {raw_election_2016.shape[1]} columns")
print(f"2020: {raw_election_2020.shape[0]:,} rows, {raw_election_2020.shape[1]} columns")
print(f"2024: {raw_election_2024.shape[0]:,} rows, {raw_election_2024.shape[1]} columns")
print("=" * 70)


ELECTION DATA LOADED
2016: 128,859 rows, 7 columns
2020: 132,123 rows, 8 columns
2024: 190,270 rows, 8 columns


/var/folders/59/n7h32tsj5s33rc_rv4x9mh6c0000gn/T/ipykernel_33188/1561404158.py:8: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_election_2020 = pd.read_csv(


In [4]:
# Load Census Data - 2016 (ACS 2012-2016)

raw_census_2016_income = pd.read_csv(DATA_PATH + 'MO 2016 Census Income.csv', keep_default_na=False, na_values=[''])
raw_census_2016_education = pd.read_csv(DATA_PATH + 'MO 2016 Census Education.csv', keep_default_na=False, na_values=[''])
raw_census_2016_race = pd.read_csv(DATA_PATH + 'MO 2016 Census Race.csv', keep_default_na=False, na_values=[''])
raw_census_2016_commute = pd.read_csv(DATA_PATH + 'MO 2016 Census Commute.csv', keep_default_na=False, na_values=[''])
raw_census_2016_sex_age = pd.read_csv(DATA_PATH + 'MO 2016 Census Sex by Age.csv', keep_default_na=False, na_values=[''])

print("=" * 70)
print("CENSUS 2016 DATA LOADED")
print("=" * 70)
print(f"Income:    {raw_census_2016_income.shape[0]:,} rows")
print(f"Education: {raw_census_2016_education.shape[0]:,} rows")
print(f"Race:      {raw_census_2016_race.shape[0]:,} rows")
print(f"Commute:   {raw_census_2016_commute.shape[0]:,} rows")
print(f"Sex/Age:   {raw_census_2016_sex_age.shape[0]:,} rows")
print("=" * 70)


CENSUS 2016 DATA LOADED
Income:    117 rows
Education: 117 rows
Race:      117 rows
Commute:   117 rows
Sex/Age:   117 rows


In [5]:
# Load Census Data - 2020 (ACS 2016-2020)

raw_census_2020_income = pd.read_csv(DATA_PATH + 'MO 2020 Census Income.csv', keep_default_na=False, na_values=[''])
raw_census_2020_education = pd.read_csv(DATA_PATH + 'MO 2020 Census Education.csv', keep_default_na=False, na_values=[''])
raw_census_2020_race = pd.read_csv(DATA_PATH + 'MO 2020 Census Race.csv', keep_default_na=False, na_values=[''])
raw_census_2020_commute = pd.read_csv(DATA_PATH + 'MO 2020 Census Commute.csv', keep_default_na=False, na_values=[''])
raw_census_2020_sex_age = pd.read_csv(DATA_PATH + 'MO 2020 Census Sex by Age.csv', keep_default_na=False, na_values=[''])

print("=" * 70)
print("CENSUS 2020 DATA LOADED")
print("=" * 70)
print(f"Income:    {raw_census_2020_income.shape[0]:,} rows")
print(f"Education: {raw_census_2020_education.shape[0]:,} rows")
print(f"Race:      {raw_census_2020_race.shape[0]:,} rows")
print(f"Commute:   {raw_census_2020_commute.shape[0]:,} rows")
print(f"Sex/Age:   {raw_census_2020_sex_age.shape[0]:,} rows")
print("=" * 70)


CENSUS 2020 DATA LOADED
Income:    117 rows
Education: 116 rows
Race:      116 rows
Commute:   117 rows
Sex/Age:   116 rows


In [6]:
# Load Census Data - 2024 (ACS 2020-2024)

raw_census_2024_income = pd.read_csv(DATA_PATH + 'MO 2024 Census Income.csv', keep_default_na=False, na_values=[''])
raw_census_2024_education = pd.read_csv(DATA_PATH + 'MO 2024 Census Education.csv', keep_default_na=False, na_values=[''])
raw_census_2024_race = pd.read_csv(DATA_PATH + 'MO 2024 Census Race.csv', keep_default_na=False, na_values=[''])
raw_census_2024_commute = pd.read_csv(DATA_PATH + 'MO 2024 Census Commute.csv', keep_default_na=False, na_values=[''])
raw_census_2024_sex_age = pd.read_csv(DATA_PATH + 'MO 2024 Census Sex by Age.csv', keep_default_na=False, na_values=[''])

print("=" * 70)
print("CENSUS 2024 DATA LOADED")
print("=" * 70)
print(f"Income:    {raw_census_2024_income.shape[0]:,} rows")
print(f"Education: {raw_census_2024_education.shape[0]:,} rows")
print(f"Race:      {raw_census_2024_race.shape[0]:,} rows")
print(f"Commute:   {raw_census_2024_commute.shape[0]:,} rows")
print(f"Sex/Age:   {raw_census_2024_sex_age.shape[0]:,} rows")
print("=" * 70)


CENSUS 2024 DATA LOADED
Income:    117 rows
Education: 117 rows
Race:      117 rows
Commute:   117 rows
Sex/Age:   116 rows


In [7]:
# Load Polling Locations Data

raw_polling_locations = pd.read_csv(
    DATA_PATH + 'MO 2020 Polling Locations.csv', 
    keep_default_na=False, 
    na_values=['']
)

print("=" * 70)
print("POLLING LOCATIONS DATA LOADED")
print("=" * 70)
print(f"Rows: {raw_polling_locations.shape[0]:,}")
print(f"Columns: {raw_polling_locations.shape[1]}")
print("=" * 70)


POLLING LOCATIONS DATA LOADED
Rows: 14,354
Columns: 15


In [8]:
# Load Shapefile Data (Precinct Boundaries)

precincts_gdf = gpd.read_file(DATA_PATH + 'MO 2020 Precincts.shp')

print("=" * 70)
print("SHAPEFILE LOADED SUCCESSFULLY")
print("=" * 70)
print(f"Total precincts: {len(precincts_gdf):,}")
print(f"CRS: {precincts_gdf.crs}")
print(f"Columns: {list(precincts_gdf.columns)}")
print("=" * 70)


SHAPEFILE LOADED SUCCESSFULLY
Total precincts: 4,604
CRS: EPSG:4269
Columns: ['STATEFP20', 'COUNTYFP20', 'VTDST20', 'GEOID20', 'VTDI20', 'NAME20', 'NAMELSAD20', 'LSAD20', 'MTFCC20', 'FUNCSTAT20', 'ALAND20', 'AWATER20', 'INTPTLAT20', 'INTPTLON20', 'geometry']


In [9]:
# Library Imports and Snowflake Connection (VSCode version)

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

# Connect to Snowflake
import snowflake.connector

conn = snowflake.connector.connect(
    account='ytdayoz-nc71712',
    user='5576GROUP1',  # <-- replace with your username
    password='Voter!Group1Final',  # <-- replace with your password
    database='VOTER_PROJECT_DB',
    schema='ANALYTICS',
    warehouse='COMPUTE_WH'
)

# Create a cursor for running queries
cursor = conn.cursor()

print("=" * 70)
print("CONNECTED TO SNOWFLAKE")
print("=" * 70)
print("Database:  VOTER_PROJECT_DB")
print("Schema:    ANALYTICS")
print("Warehouse: COMPUTE_WH")
print("=" * 70)

CONNECTED TO SNOWFLAKE
Database:  VOTER_PROJECT_DB
Schema:    ANALYTICS
Warehouse: COMPUTE_WH


In [10]:
import geopandas as gpd
print(f"geopandas version: {gpd.__version__}")

geopandas version: 1.1.3


---
# SECTION 2: Data Ingestion
---

Load all raw data files into pandas DataFrames. Files are loaded exactly as downloaded to preserve raw source data. All transformations occur programmatically to ensure reproducibility.

---
# SECTION 3: Data Cleaning - Election Data
---

Clean and standardize election results across all three years:
1. Add year column
2. Drop precinct_code (2020/2024 only)
3. Normalize precinct and county names
4. Filter to Presidential results only
5. Combine into single dataset

In [11]:
# Define Precinct Name Normalization Function

def normalize_precinct_name(name):
    """
    Standardize precinct names for consistent joining across datasets.
    Missouri precinct names vary (e.g., 'Ward 1', 'WARD 1', 'Ward #1').
    """
    return (
        str(name)
        .upper()
        .replace('#', '')
        .replace('  ', ' ')
        .strip()
    )

print("normalize_precinct_name() function defined")

normalize_precinct_name() function defined


In [12]:
# Clean and Filter Election Results (Presidential Only)

election_2016_clean = (
    raw_election_2016
    .query("office == 'President'")
    .assign(year=2016)
    .assign(precinct_clean=lambda x: x['precinct'].apply(normalize_precinct_name))
    .assign(county_clean=lambda x: x['county'].str.upper().str.strip())
    [['year', 'county', 'county_clean', 'precinct', 'precinct_clean', 
      'office', 'district', 'candidate', 'party', 'votes']]
    .reset_index(drop=True)
)

election_2020_clean = (
    raw_election_2020
    .query("office == 'President'")
    .drop(columns=['precinct_code'], errors='ignore')
    .assign(year=2020)
    .assign(precinct_clean=lambda x: x['precinct'].apply(normalize_precinct_name))
    .assign(county_clean=lambda x: x['county'].str.upper().str.strip())
    [['year', 'county', 'county_clean', 'precinct', 'precinct_clean', 
      'office', 'district', 'candidate', 'party', 'votes']]
    .reset_index(drop=True)
)

election_2024_clean = (
    raw_election_2024
    .loc[lambda x: x['office'] == 'President']
    .drop(columns=['precinct_code'], errors='ignore')
    .assign(year=2024)
    .assign(precinct_clean=lambda x: x['precinct'].apply(normalize_precinct_name))
    .assign(county_clean=lambda x: x['county'].str.upper().str.strip())
    [['year', 'county', 'county_clean', 'precinct', 'precinct_clean', 
      'office', 'district', 'candidate', 'party', 'votes']]
    .reset_index(drop=True)
)

print("=" * 70)
print("PRESIDENTIAL ELECTION DATA CLEANED")
print("=" * 70)
print(f"2016 Presidential: {election_2016_clean.shape[0]:>8,} rows")
print(f"2020 Presidential: {election_2020_clean.shape[0]:>8,} rows")
print(f"2024 Presidential: {election_2024_clean.shape[0]:>8,} rows")
print("=" * 70)

PRESIDENTIAL ELECTION DATA CLEANED
2016 Presidential:   15,880 rows
2020 Presidential:   25,689 rows
2024 Presidential:   28,180 rows


In [13]:
# Combine All Election Years into Single Dataset

all_elections_df = pd.concat(
    [election_2016_clean, election_2020_clean, election_2024_clean], 
    ignore_index=True
)

print("=" * 70)
print("COMBINED ELECTION DATASET")
print("=" * 70)
print(f"Total Rows:    {all_elections_df.shape[0]:,}")
print(f"Total Columns: {all_elections_df.shape[1]}")
print(f"Years:         {sorted(all_elections_df['year'].unique().tolist())}")
print(f"Counties:      {all_elections_df['county_clean'].nunique()}")
print(f"Parties:       {all_elections_df['party'].unique().tolist()}")
print("=" * 70)
all_elections_df.head(10)

COMBINED ELECTION DATASET
Total Rows:    69,749
Total Columns: 10
Years:         [2016, 2020, 2024]
Counties:      117
Parties:       ['DEM', 'REP', 'LBT', 'CON', 'GTN', 'WRITE-IN', 'LIB', 'GRE', 'CST', 'WI', nan, 'GRN']


,year,county,county_clean,precinct,precinct_clean,office,district,candidate,party,votes
0,2016,Adair,ADAIR,SOUTHWEST ONE/BENTON,SOUTHWEST ONE/BENTON,President,NaN,"Hillary Rodham Clinton, Timothy Michael Kaine",DEM,224.0
1,2016,Adair,ADAIR,SOUTHEAST TWO/BENTON,SOUTHEAST TWO/BENTON,President,NaN,"Hillary Rodham Clinton, Timothy Michael Kaine",DEM,458.0
2,2016,Adair,ADAIR,SOUTHEAST THREE/BENTON,SOUTHEAST THREE/BENTON,President,NaN,"Hillary Rodham Clinton, Timothy Michael Kaine",DEM,391.0
3,2016,Adair,ADAIR,NORTHEAST FOUR/BENTON,NORTHEAST FOUR/BENTON,President,NaN,"Hillary Rodham Clinton, Timothy Michael Kaine",DEM,386.0
4,2016,Adair,ADAIR,NORTHEAST FIVE/BENTON,NORTHEAST FIVE/BENTON,President,NaN,"Hillary Rodham Clinton, Timothy Michael Kaine",DEM,256.0
5,2016,Adair,ADAIR,NORTHEAST SIX/BENTON,NORTHEAST SIX/BENTON,President,NaN,"Hillary Rodham Clinton, Timothy Michael Kaine",DEM,317.0
6,2016,Adair,ADAIR,TSU/BENTON,TSU/BENTON,President,NaN,"Hillary Rodham Clinton, Timothy Michael Kaine",DEM,281.0
7,2016,Adair,ADAIR,RURAL BENTON/BENTON,RURAL BENTON/BENTON,President,NaN,"Hillary Rodham Clinton, Timothy Michael Kaine",DEM,359.0
8,2016,Adair,ADAIR,NOVINGER,NOVINGER,President,NaN,"Hillary Rodham Clinton, Timothy Michael Kaine",DEM,179.0
9,2016,Adair,ADAIR,BRASHEAR,BRASHEAR,President,NaN,"Hillary Rodham Clinton, Timothy Michael Kaine",DEM,143.0


---
# SECTION 4: Data Cleaning - Census Data
---

Clean census data for all three years. Each census year is processed identically:
1. Remove header artifact row (GEO_ID = "Geography")
2. Remove state-level totals
3. Extract county FIPS and clean county name
4. Add census year column
5. Rename ACS codes to readable names
6. Convert to numeric types
7. Calculate derived metrics (percentages)

In [14]:
# Define Census Cleaning Functions

# NOTE: We do NOT remove " city" from county names to preserve the distinction
# between St. Louis County (FIPS 29189) and St. Louis city (FIPS 29510)

def clean_census_income(df, census_year):
    """Clean census income data for a given year."""
    return (
        df
        .query("GEO_ID != 'Geography'")
        .query("GEO_ID.str.contains('0500000US', na=False)")
        .assign(census_year=census_year)
        .assign(county_fips=lambda x: x['GEO_ID'].str[-5:])
        .assign(county_name=lambda x: x['NAME'].str.replace(', Missouri', '').str.replace(' County', ''))
        .assign(county_clean=lambda x: x['county_name'].str.upper().str.strip())
        .rename(columns={'B19013_001E': 'median_household_income'})
        .assign(median_household_income=lambda x: pd.to_numeric(x['median_household_income'], errors='coerce'))
        [['census_year', 'county_fips', 'county_name', 'county_clean', 'median_household_income']]
        .reset_index(drop=True)
    )

def clean_census_education(df, census_year):
    """Clean census education data for a given year."""
    result = (
        df
        .query("GEO_ID != 'Geography'")
        .query("GEO_ID.str.contains('0500000US', na=False)")
        .assign(census_year=census_year)
        .assign(county_fips=lambda x: x['GEO_ID'].str[-5:])
        .assign(county_name=lambda x: x['NAME'].str.replace(', Missouri', '').str.replace(' County', ''))
        .assign(county_clean=lambda x: x['county_name'].str.upper().str.strip())
        .rename(columns={
            'B15003_001E': 'total_pop_25_plus',
            'B15003_017E': 'hs_diploma',
            'B15003_022E': 'bachelors_degree',
            'B15003_023E': 'masters_degree',
            'B15003_024E': 'professional_degree',
            'B15003_025E': 'doctorate_degree'
        })
        .reset_index(drop=True)
    )
    
    for col in ['total_pop_25_plus', 'hs_diploma', 'bachelors_degree', 'masters_degree', 'professional_degree', 'doctorate_degree']:
        if col in result.columns:
            result[col] = pd.to_numeric(result[col], errors='coerce')
    
    result['pct_bachelors_plus'] = (
        (result['bachelors_degree'] + result['masters_degree'] + 
         result['professional_degree'] + result['doctorate_degree']) 
        / result['total_pop_25_plus'] * 100
    ).round(2)
    
    return result[['census_year', 'county_fips', 'county_name', 'county_clean', 
                   'total_pop_25_plus', 'pct_bachelors_plus']]

def clean_census_race(df, census_year):
    """Clean census race data for a given year."""
    result = (
        df
        .query("GEO_ID != 'Geography'")
        .query("GEO_ID.str.contains('0500000US', na=False)")
        .assign(census_year=census_year)
        .assign(county_fips=lambda x: x['GEO_ID'].str[-5:])
        .assign(county_name=lambda x: x['NAME'].str.replace(', Missouri', '').str.replace(' County', ''))
        .assign(county_clean=lambda x: x['county_name'].str.upper().str.strip())
        .rename(columns={
            'B02001_001E': 'total_population',
            'B02001_002E': 'white_alone',
            'B02001_003E': 'black_alone'
        })
        .reset_index(drop=True)
    )
    
    for col in ['total_population', 'white_alone', 'black_alone']:
        if col in result.columns:
            result[col] = pd.to_numeric(result[col], errors='coerce')
    
    result['pct_white'] = (result['white_alone'] / result['total_population'] * 100).round(2)
    result['pct_minority'] = ((result['total_population'] - result['white_alone']) / result['total_population'] * 100).round(2)
    
    return result[['census_year', 'county_fips', 'county_name', 'county_clean', 
                   'total_population', 'pct_white', 'pct_minority']]

def clean_census_commute(df, census_year):
    """Clean census commute data for a given year."""
    result = (
        df
        .query("GEO_ID != 'Geography'")
        .query("GEO_ID.str.contains('0500000US', na=False)")
        .assign(census_year=census_year)
        .assign(county_fips=lambda x: x['GEO_ID'].str[-5:])
        .assign(county_name=lambda x: x['NAME'].str.replace(', Missouri', '').str.replace(' County', ''))
        .assign(county_clean=lambda x: x['county_name'].str.upper().str.strip())
        .rename(columns={
            'B08301_001E': 'total_workers',
            'B08301_003E': 'drove_alone',
            'B08301_010E': 'public_transit',
            'B08301_019E': 'walked'
        })
        .reset_index(drop=True)
    )
    
    for col in ['total_workers', 'drove_alone', 'public_transit', 'walked']:
        if col in result.columns:
            result[col] = pd.to_numeric(result[col], errors='coerce')
    
    result['pct_no_vehicle'] = ((result['public_transit'] + result['walked']) / result['total_workers'] * 100).round(2)
    
    return result[['census_year', 'county_fips', 'county_name', 'county_clean', 
                   'total_workers', 'pct_no_vehicle']]

def clean_census_sex_age(df, census_year):
    """Clean census sex by age data for a given year."""
    result = (
        df
        .query("GEO_ID != 'Geography'")
        .query("GEO_ID.str.contains('0500000US', na=False)")
        .assign(census_year=census_year)
        .assign(county_fips=lambda x: x['GEO_ID'].str[-5:])
        .assign(county_name=lambda x: x['NAME'].str.replace(', Missouri', '').str.replace(' County', ''))
        .assign(county_clean=lambda x: x['county_name'].str.upper().str.strip())
        .reset_index(drop=True)
    )
    
    for col in result.columns:
        if col.startswith('B01001'):
            result[col] = pd.to_numeric(result[col], errors='coerce')
    
    result['total_population'] = result['B01001_001E']
    
    male_18plus_cols = [f'B01001_{str(i).zfill(3)}E' for i in range(7, 26)]
    female_18plus_cols = [f'B01001_{str(i).zfill(3)}E' for i in range(31, 50)]
    
    male_18plus_cols = [c for c in male_18plus_cols if c in result.columns]
    female_18plus_cols = [c for c in female_18plus_cols if c in result.columns]
    
    result['voting_age_population'] = result[male_18plus_cols].sum(axis=1) + result[female_18plus_cols].sum(axis=1)
    result['pct_voting_age'] = (result['voting_age_population'] / result['total_population'] * 100).round(2)
    
    return result[['census_year', 'county_fips', 'county_name', 'county_clean', 
                   'total_population', 'voting_age_population', 'pct_voting_age']]

print("Census cleaning functions defined")

Census cleaning functions defined


In [15]:
# Clean Census Income Data (All Years)

census_income_2016 = clean_census_income(raw_census_2016_income, 2016)
census_income_2020 = clean_census_income(raw_census_2020_income, 2020)
census_income_2024 = clean_census_income(raw_census_2024_income, 2024)

all_census_income_df = pd.concat(
    [census_income_2016, census_income_2020, census_income_2024],
    ignore_index=True
)

print(f"Census Income cleaned: {all_census_income_df.shape[0]} rows ({all_census_income_df['census_year'].nunique()} years)")
all_census_income_df.head()

Census Income cleaned: 345 rows (3 years)


,census_year,county_fips,county_name,county_clean,median_household_income
0,2016,29001,Adair,ADAIR,37967
1,2016,29003,Andrew,ANDREW,54804
2,2016,29005,Atchison,ATCHISON,43438
3,2016,29007,Audrain,AUDRAIN,41930
4,2016,29009,Barry,BARRY,37869


In [16]:
# Clean Census Education Data (All Years)

census_education_2016 = clean_census_education(raw_census_2016_education, 2016)
census_education_2020 = clean_census_education(raw_census_2020_education, 2020)
census_education_2024 = clean_census_education(raw_census_2024_education, 2024)

all_census_education_df = pd.concat(
    [census_education_2016, census_education_2020, census_education_2024],
    ignore_index=True
)

print(f"Census Education cleaned: {all_census_education_df.shape[0]} rows ({all_census_education_df['census_year'].nunique()} years)")
all_census_education_df.head()

Census Education cleaned: 345 rows (3 years)


,census_year,county_fips,county_name,county_clean,total_pop_25_plus,pct_bachelors_plus
0,2016,29001,Adair,ADAIR,13621,30.49
1,2016,29003,Andrew,ANDREW,12005,23.98
2,2016,29005,Atchison,ATCHISON,3945,22.33
3,2016,29007,Audrain,AUDRAIN,17659,12.59
4,2016,29009,Barry,BARRY,24688,12.81


In [17]:
# Clean Census Race Data (All Years)

census_race_2016 = clean_census_race(raw_census_2016_race, 2016)
census_race_2020 = clean_census_race(raw_census_2020_race, 2020)
census_race_2024 = clean_census_race(raw_census_2024_race, 2024)

all_census_race_df = pd.concat(
    [census_race_2016, census_race_2020, census_race_2024],
    ignore_index=True
)

print(f"Census Race cleaned: {all_census_race_df.shape[0]} rows ({all_census_race_df['census_year'].nunique()} years)")
all_census_race_df.head()

Census Race cleaned: 345 rows (3 years)


,census_year,county_fips,county_name,county_clean,total_population,pct_white,pct_minority
0,2016,29001,Adair,ADAIR,25547,92.57,7.43
1,2016,29003,Andrew,ANDREW,17347,96.22,3.78
2,2016,29005,Atchison,ATCHISON,5380,97.83,2.17
3,2016,29007,Audrain,AUDRAIN,25868,89.10,10.90
4,2016,29009,Barry,BARRY,35716,92.96,7.04


In [18]:
# Clean Census Commute Data (All Years)

census_commute_2016 = clean_census_commute(raw_census_2016_commute, 2016)
census_commute_2020 = clean_census_commute(raw_census_2020_commute, 2020)
census_commute_2024 = clean_census_commute(raw_census_2024_commute, 2024)

all_census_commute_df = pd.concat(
    [census_commute_2016, census_commute_2020, census_commute_2024],
    ignore_index=True
)

print(f"Census Commute cleaned: {all_census_commute_df.shape[0]} rows ({all_census_commute_df['census_year'].nunique()} years)")
all_census_commute_df.head()

Census Commute cleaned: 345 rows (3 years)


,census_year,county_fips,county_name,county_clean,total_workers,pct_no_vehicle
0,2016,29001,Adair,ADAIR,10849,4.53
1,2016,29003,Andrew,ANDREW,8397,1.14
2,2016,29005,Atchison,ATCHISON,2597,3.43
3,2016,29007,Audrain,AUDRAIN,10584,3.69
4,2016,29009,Barry,BARRY,13885,2.53


In [19]:
# Clean Census Sex by Age Data (All Years)

census_sex_age_2016 = clean_census_sex_age(raw_census_2016_sex_age, 2016)
census_sex_age_2020 = clean_census_sex_age(raw_census_2020_sex_age, 2020)
census_sex_age_2024 = clean_census_sex_age(raw_census_2024_sex_age, 2024)

all_census_sex_age_df = pd.concat(
    [census_sex_age_2016, census_sex_age_2020, census_sex_age_2024],
    ignore_index=True
)

print(f"Census Sex/Age cleaned: {all_census_sex_age_df.shape[0]} rows ({all_census_sex_age_df['census_year'].nunique()} years)")
all_census_sex_age_df.head()

Census Sex/Age cleaned: 345 rows (3 years)


,census_year,county_fips,county_name,county_clean,total_population,voting_age_population,pct_voting_age
0,2016,29001,Adair,ADAIR,25547,20817,81.49
1,2016,29003,Andrew,ANDREW,17347,13355,76.99
2,2016,29005,Atchison,ATCHISON,5380,4339,80.65
3,2016,29007,Audrain,AUDRAIN,25868,19855,76.76
4,2016,29009,Barry,BARRY,35716,27535,77.09


---
# SECTION 5: Data Cleaning - Polling Locations & Shapefile
---

Clean polling locations and prepare shapefile for potential geospatial analysis.

In [20]:
# Clean Polling Locations Data

polling_locations_df = (
    raw_polling_locations
    .copy()
    .assign(county_clean=lambda x: x['county_name'].str.strip().str.upper())
    .assign(precinct_clean=lambda x: x['precinct_name'].apply(normalize_precinct_name))
    .assign(polling_address=lambda x: x['address'].str.strip())
)

print("=" * 70)
print("POLLING LOCATIONS CLEANED")
print("=" * 70)
print(f"Total Rows:           {polling_locations_df.shape[0]:,}")
print(f"Unique Counties:      {polling_locations_df['county_clean'].nunique()}")
print(f"Unique Precincts:     {polling_locations_df['precinct_clean'].nunique()}")
print(f"Unique Polling Places: {polling_locations_df['polling_place_id'].nunique()}")
print("=" * 70)
polling_locations_df.head()

POLLING LOCATIONS CLEANED
Total Rows:           14,354
Unique Counties:      116
Unique Precincts:     13913
Unique Polling Places: 2133


,election_date,state,county_name,jurisdiction,jurisdiction_type,precinct_id,precinct_name,polling_place_id,location_type,name,address,notes,source,source_date,source_notes,county_clean,precinct_clean,polling_address
0,2020-11-03,MO,Adair,Adair,county,1009-03,BRASHEAR/WILSON 03,BRASHEAR1,NaN,NEMO FAIR GROUNDS,"2700 E ILLINOIS ST KIRKSVILLE, 63501",NaN,website,2020-10-06,NaN,ADAIR,BRASHEAR/WILSON 03,"2700 E ILLINOIS ST KIRKSVILLE, 63501"
1,2020-11-03,MO,Adair,Adair,county,1009-04,BRASHEAR/WILSON 04,BRASHEAR1,NaN,NEMO FAIR GROUNDS,"2700 E ILLINOIS ST KIRKSVILLE, 63501",NaN,website,2020-10-06,NaN,ADAIR,BRASHEAR/WILSON 04,"2700 E ILLINOIS ST KIRKSVILLE, 63501"
2,2020-11-03,MO,Adair,Adair,county,1009-05,BRASHEAR/WILSON 05,BRASHEAR1,NaN,NEMO FAIR GROUNDS,"2700 E ILLINOIS ST KIRKSVILLE, 63501",NaN,website,2020-10-06,NaN,ADAIR,BRASHEAR/WILSON 05,"2700 E ILLINOIS ST KIRKSVILLE, 63501"
3,2020-11-03,MO,Adair,Adair,county,1009-07,BRASHEAR/WILSON 07,BRASHEAR1,NaN,NEMO FAIR GROUNDS,"2700 E ILLINOIS ST KIRKSVILLE, 63501",NaN,website,2020-10-06,NaN,ADAIR,BRASHEAR/WILSON 07,"2700 E ILLINOIS ST KIRKSVILLE, 63501"
4,2020-11-03,MO,Adair,Adair,county,1009-08,BRASHEAR/WILSON 08,BRASHEAR1,NaN,NEMO FAIR GROUNDS,"2700 E ILLINOIS ST KIRKSVILLE, 63501",NaN,website,2020-10-06,NaN,ADAIR,BRASHEAR/WILSON 08,"2700 E ILLINOIS ST KIRKSVILLE, 63501"


In [21]:
# Prepare Shapefile Data
# David, this could will not work until we get geopandas imported above
#   So, I've commented it out

# precincts_df = (
#     raw_precincts_gdf
#     .drop(columns=['geometry'])
#     .assign(county_fips=lambda x: '29' + x['COUNTYFP20'])
#     .assign(precinct_name=lambda x: x['NAME20'])
#     .assign(precinct_clean=lambda x: x['NAME20'].apply(normalize_precinct_name))
#     .assign(land_area_sqm=lambda x: x['ALAND20'])
#     .assign(water_area_sqm=lambda x: x['AWATER20'])
#     .assign(centroid_lat=lambda x: pd.to_numeric(x['INTPTLAT20'], errors='coerce'))
#     .assign(centroid_lon=lambda x: pd.to_numeric(x['INTPTLON20'], errors='coerce'))
#     [['county_fips', 'COUNTYFP20', 'precinct_name', 'precinct_clean', 
#       'GEOID20', 'land_area_sqm', 'water_area_sqm', 'centroid_lat', 'centroid_lon']]
#     .rename(columns={'COUNTYFP20': 'county_fips_3', 'GEOID20': 'geoid'})
# )

# print("=" * 70)
# print("SHAPEFILE DATA PREPARED (Tabular - No Geometry)")
# print("=" * 70)
# print(f"Precincts: {precincts_df.shape[0]:,}")
# print(f"Columns:   {precincts_df.columns.tolist()}")
# print("=" * 70)
# precincts_df.head()

In [22]:
!pip install "snowflake-connector-python[pandas]"
!pip install pyarrow

---
# SECTION 6: Write Staging Tables to Snowflake
---

Write all cleaned DataFrames to Snowflake staging tables. These tables will be used for SQL-based aggregations and joins.

In [23]:
# Write All Staging Tables to Snowflake
from snowflake.connector.pandas_tools import write_pandas

staging_tables = {
    'STG_ELECTION_RESULTS': all_elections_df,
    'STG_CENSUS_INCOME': all_census_income_df,
    'STG_CENSUS_EDUCATION': all_census_education_df,
    'STG_CENSUS_RACE': all_census_race_df,
    'STG_CENSUS_COMMUTE': all_census_commute_df,
    'STG_CENSUS_SEX_AGE': all_census_sex_age_df,
    'STG_POLLING_LOCATIONS': polling_locations_df
}

print("=" * 70)
print("WRITING STAGING TABLES TO SNOWFLAKE")
print("=" * 70)

for table_name, df in staging_tables.items():
    # Write DataFrame to Snowflake
    success, nchunks, nrows, _ = write_pandas(
        conn, 
        df, 
        table_name, 
        auto_create_table=True,
        overwrite=True
    )
    print(f"{table_name}: {nrows:,} rows")

print("=" * 70)
print("ALL STAGING TABLES WRITTEN")
print("=" * 70)

WRITING STAGING TABLES TO SNOWFLAKE
STG_ELECTION_RESULTS: 69,749 rows
STG_CENSUS_INCOME: 345 rows
STG_CENSUS_EDUCATION: 345 rows
STG_CENSUS_RACE: 345 rows
STG_CENSUS_COMMUTE: 345 rows
STG_CENSUS_SEX_AGE: 345 rows
STG_POLLING_LOCATIONS: 14,354 rows
ALL STAGING TABLES WRITTEN


## Snowflake Column Naming Convention

**Staging tables** (created via Python) have **lowercase** column names:
- Reference with double quotes in SQL: `"year"`, `"county_clean"`

**Analytical tables** (created via SQL) use **UPPERCASE** aliases:
- Created with: `SELECT "year" AS YEAR`
- Reference without quotes: `WHERE YEAR = 2020`

This convention simplifies downstream SQL queries.

---
# SECTION 7: SQL Aggregations & JOINs (Declarative)
---

Create analytical tables using SQL:
- Aggregate precinct-level votes to county level
- Pivot turnout by year
- Join election results with census demographics

In [24]:
# Execute SQL: Create/Update Table
sql = """
CREATE OR REPLACE TABLE PRECINCT_TURNOUT AS
SELECT 
    "year" AS YEAR,
    "county_clean" AS COUNTY,
    "precinct_clean" AS PRECINCT,
    SUM("votes") AS TOTAL_VOTES,
    SUM(CASE WHEN "party" = 'REP' THEN "votes" ELSE 0 END) AS REPUBLICAN_VOTES,
    SUM(CASE WHEN "party" = 'DEM' THEN "votes" ELSE 0 END) AS DEMOCRAT_VOTES,
    SUM(CASE WHEN "party" NOT IN ('REP', 'DEM') OR "party" IS NULL THEN "votes" ELSE 0 END) AS OTHER_VOTES,
    ROUND(SUM(CASE WHEN "party" = 'REP' THEN "votes" ELSE 0 END) / NULLIF(SUM("votes"), 0) * 100, 2) AS REPUBLICAN_PCT,
    ROUND(SUM(CASE WHEN "party" = 'DEM' THEN "votes" ELSE 0 END) / NULLIF(SUM("votes"), 0) * 100, 2) AS DEMOCRAT_PCT
FROM STG_ELECTION_RESULTS
GROUP BY "year", "county_clean", "precinct_clean"
ORDER BY "year", "county_clean", "precinct_clean";
"""
cursor.execute(sql)
print("Table created/updated successfully")

Table created/updated successfully


In [25]:
# Execute SQL: Create/Update Table
sql = """
-- Aggregate to county level for each year
CREATE OR REPLACE TABLE COUNTY_TURNOUT AS
SELECT 
    YEAR,
    COUNTY,
    COUNT(DISTINCT PRECINCT) AS PRECINCT_COUNT,
    SUM(TOTAL_VOTES) AS TOTAL_VOTES,
    SUM(REPUBLICAN_VOTES) AS REPUBLICAN_VOTES,
    SUM(DEMOCRAT_VOTES) AS DEMOCRAT_VOTES,
    SUM(OTHER_VOTES) AS OTHER_VOTES,
    ROUND(SUM(REPUBLICAN_VOTES) / NULLIF(SUM(TOTAL_VOTES), 0) * 100, 2) AS REPUBLICAN_PCT,
    ROUND(SUM(DEMOCRAT_VOTES) / NULLIF(SUM(TOTAL_VOTES), 0) * 100, 2) AS DEMOCRAT_PCT
FROM PRECINCT_TURNOUT
GROUP BY YEAR, COUNTY
ORDER BY YEAR, COUNTY;
"""
cursor.execute(sql)
print("Table created/updated successfully")

Table created/updated successfully


In [26]:
# Execute SQL: Create/Update Table
sql = """
-- Pivot to show all years side by side for trend analysis
CREATE OR REPLACE TABLE COUNTY_TURNOUT_TREND AS
SELECT 
    COUNTY,
    MAX(CASE WHEN YEAR = 2016 THEN PRECINCT_COUNT END) AS PRECINCTS_2016,
    MAX(CASE WHEN YEAR = 2020 THEN PRECINCT_COUNT END) AS PRECINCTS_2020,
    MAX(CASE WHEN YEAR = 2024 THEN PRECINCT_COUNT END) AS PRECINCTS_2024,
    MAX(CASE WHEN YEAR = 2016 THEN TOTAL_VOTES END) AS VOTES_2016,
    MAX(CASE WHEN YEAR = 2020 THEN TOTAL_VOTES END) AS VOTES_2020,
    MAX(CASE WHEN YEAR = 2024 THEN TOTAL_VOTES END) AS VOTES_2024,
    MAX(CASE WHEN YEAR = 2016 THEN REPUBLICAN_PCT END) AS REP_PCT_2016,
    MAX(CASE WHEN YEAR = 2020 THEN REPUBLICAN_PCT END) AS REP_PCT_2020,
    MAX(CASE WHEN YEAR = 2024 THEN REPUBLICAN_PCT END) AS REP_PCT_2024,
    MAX(CASE WHEN YEAR = 2016 THEN DEMOCRAT_PCT END) AS DEM_PCT_2016,
    MAX(CASE WHEN YEAR = 2020 THEN DEMOCRAT_PCT END) AS DEM_PCT_2020,
    MAX(CASE WHEN YEAR = 2024 THEN DEMOCRAT_PCT END) AS DEM_PCT_2024
FROM COUNTY_TURNOUT
GROUP BY COUNTY
ORDER BY COUNTY;
"""
cursor.execute(sql)
print("Table created/updated successfully")

Table created/updated successfully


In [27]:
# Execute SQL: Create/Update Table
sql = """
-- Combine all census tables and pivot to get one row per county with all years
CREATE OR REPLACE TABLE COUNTY_CENSUS AS
SELECT 
    i."county_clean" AS COUNTY,
    i."county_fips" AS COUNTY_FIPS,
    i."county_name" AS COUNTY_NAME,
    i."census_year" AS CENSUS_YEAR,
    i."median_household_income" AS MEDIAN_HOUSEHOLD_INCOME,
    e."pct_bachelors_plus" AS PCT_BACHELORS_PLUS,
    r."total_population" AS TOTAL_POPULATION,
    r."pct_minority" AS PCT_MINORITY,
    c."pct_no_vehicle" AS PCT_NO_VEHICLE,
    s."voting_age_population" AS VOTING_AGE_POPULATION,
    s."pct_voting_age" AS PCT_VOTING_AGE
FROM STG_CENSUS_INCOME i
LEFT JOIN STG_CENSUS_EDUCATION e 
    ON i."county_clean" = e."county_clean" AND i."census_year" = e."census_year"
LEFT JOIN STG_CENSUS_RACE r 
    ON i."county_clean" = r."county_clean" AND i."census_year" = r."census_year"
LEFT JOIN STG_CENSUS_COMMUTE c 
    ON i."county_clean" = c."county_clean" AND i."census_year" = c."census_year"
LEFT JOIN STG_CENSUS_SEX_AGE s 
    ON i."county_clean" = s."county_clean" AND i."census_year" = s."census_year"
ORDER BY i."county_clean", i."census_year";
"""
cursor.execute(sql)
print("Table created/updated successfully")

Table created/updated successfully


In [28]:
# Execute SQL: Create/Update Table
sql = """
-- Join turnout trends with census data for comprehensive analysis table
CREATE OR REPLACE TABLE COUNTY_ANALYSIS AS
SELECT 
    t.COUNTY,
    
    -- Turnout by year
    t.VOTES_2016,
    t.VOTES_2020,
    t.VOTES_2024,
    t.REP_PCT_2016,
    t.REP_PCT_2020,
    t.REP_PCT_2024,
    t.DEM_PCT_2016,
    t.DEM_PCT_2020,
    t.DEM_PCT_2024,
    
    -- 2016 Census
    c16.TOTAL_POPULATION AS POP_2016,
    c16.VOTING_AGE_POPULATION AS VAP_2016,
    c16.MEDIAN_HOUSEHOLD_INCOME AS INCOME_2016,
    c16.PCT_BACHELORS_PLUS AS EDU_2016,
    c16.PCT_MINORITY AS MINORITY_2016,
    
    -- 2020 Census
    c20.TOTAL_POPULATION AS POP_2020,
    c20.VOTING_AGE_POPULATION AS VAP_2020,
    c20.MEDIAN_HOUSEHOLD_INCOME AS INCOME_2020,
    c20.PCT_BACHELORS_PLUS AS EDU_2020,
    c20.PCT_MINORITY AS MINORITY_2020,
    c20.PCT_NO_VEHICLE AS NO_VEHICLE_2020,
    
    -- 2024 Census
    c24.TOTAL_POPULATION AS POP_2024,
    c24.VOTING_AGE_POPULATION AS VAP_2024,
    c24.MEDIAN_HOUSEHOLD_INCOME AS INCOME_2024,
    c24.PCT_BACHELORS_PLUS AS EDU_2024,
    c24.PCT_MINORITY AS MINORITY_2024,
    
    -- Calculated turnout rates
    ROUND(t.VOTES_2016 / NULLIF(c16.VOTING_AGE_POPULATION, 0) * 100, 2) AS TURNOUT_PCT_2016,
    ROUND(t.VOTES_2020 / NULLIF(c20.VOTING_AGE_POPULATION, 0) * 100, 2) AS TURNOUT_PCT_2020,
    ROUND(t.VOTES_2024 / NULLIF(c24.VOTING_AGE_POPULATION, 0) * 100, 2) AS TURNOUT_PCT_2024

FROM COUNTY_TURNOUT_TREND t
LEFT JOIN COUNTY_CENSUS c16 ON t.COUNTY = c16.COUNTY AND c16.CENSUS_YEAR = 2016
LEFT JOIN COUNTY_CENSUS c20 ON t.COUNTY = c20.COUNTY AND c20.CENSUS_YEAR = 2020
LEFT JOIN COUNTY_CENSUS c24 ON t.COUNTY = c24.COUNTY AND c24.CENSUS_YEAR = 2024
ORDER BY t.COUNTY;
"""
cursor.execute(sql)
print("Table created/updated successfully")

Table created/updated successfully


In [29]:
# Execute SQL: Create/Update Table
sql = """
-- Aggregate polling locations by county
CREATE OR REPLACE TABLE COUNTY_POLLING_SUMMARY AS
SELECT 
    "county_clean" AS COUNTY,
    COUNT(DISTINCT "polling_place_id") AS UNIQUE_POLLING_PLACES,
    COUNT(DISTINCT "precinct_clean") AS UNIQUE_PRECINCTS,
    COUNT(*) AS TOTAL_RECORDS,
    ROUND(COUNT(DISTINCT "precinct_clean") / NULLIF(COUNT(DISTINCT "polling_place_id"), 0), 2) AS PRECINCTS_PER_POLLING_PLACE
FROM STG_POLLING_LOCATIONS
GROUP BY "county_clean"
ORDER BY PRECINCTS_PER_POLLING_PLACE DESC;
"""
cursor.execute(sql)
print("Table created/updated successfully")

Table created/updated successfully


---
# SECTION 8: Data Quality Validation
---

Verify data integrity before proceeding with EDA:
- Row counts for all tables
- County matching validation
- Missing value checks

### Expected Row Counts

| Table | Expected Rows |
|-------|---------------|
| COUNTY_ANALYSIS | 117 (one per county) |
| COUNTY_TURNOUT | 348 (116 counties × 3 years) |
| COUNTY_CENSUS | 345 (115 counties × 3 years) |

In [30]:
# Data Quality Summary - Table Row Counts

staging_tables = [
    'STG_ELECTION_RESULTS',
    'STG_CENSUS_INCOME',
    'STG_CENSUS_EDUCATION',
    'STG_CENSUS_RACE',
    'STG_CENSUS_COMMUTE',
    'STG_CENSUS_SEX_AGE',
    'STG_POLLING_LOCATIONS'
]

analytical_tables = [
    'PRECINCT_TURNOUT',
    'COUNTY_TURNOUT',
    'COUNTY_TURNOUT_TREND',
    'COUNTY_CENSUS',
    'COUNTY_ANALYSIS',
    'COUNTY_POLLING_SUMMARY'
]

print("=" * 70)
print("DATA QUALITY SUMMARY - TABLE ROW COUNTS")
print("=" * 70)

print("\nSTAGING TABLES:")
for table in staging_tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"   {table}: {count:,} rows")

print("\nANALYTICAL TABLES:")
for table in analytical_tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"   {table}: {count:,} rows")

print("=" * 70)

DATA QUALITY SUMMARY - TABLE ROW COUNTS

STAGING TABLES:
   STG_ELECTION_RESULTS: 69,749 rows
   STG_CENSUS_INCOME: 345 rows
   STG_CENSUS_EDUCATION: 345 rows
   STG_CENSUS_RACE: 345 rows
   STG_CENSUS_COMMUTE: 345 rows
   STG_CENSUS_SEX_AGE: 345 rows
   STG_POLLING_LOCATIONS: 14,354 rows

ANALYTICAL TABLES:
   PRECINCT_TURNOUT: 10,318 rows
   COUNTY_TURNOUT: 348 rows
   COUNTY_TURNOUT_TREND: 117 rows
   COUNTY_CENSUS: 345 rows
   COUNTY_ANALYSIS: 117 rows
   COUNTY_POLLING_SUMMARY: 116 rows


In [31]:
# Execute SQL Query
sql = """
-- Check for county mismatches between election and census data

SELECT 
    'Election counties not in Census' AS validation_check,
    COUNT(DISTINCT t.county) AS count
FROM COUNTY_TURNOUT t
LEFT JOIN COUNTY_CENSUS c ON t.county = c.county
WHERE c.county IS NULL

UNION ALL

SELECT 
    'Census counties not in Election' AS validation_check,
    COUNT(DISTINCT c.county) AS count
FROM COUNTY_CENSUS c
LEFT JOIN COUNTY_TURNOUT t ON c.county = t.county
WHERE t.county IS NULL;
"""
df = pd.read_sql(sql, conn)
df

,VALIDATION_CHECK,COUNT
0,Election counties not in Census,3
1,Census counties not in Election,1


---
# SECTION 9: Exploratory Data Analysis (EDA)
---

Analyze patterns in:
- Statewide turnout trends
- County-level turnout variations
- Demographic correlations
- Polling resource distribution

In [32]:
# Execute SQL Query
sql = """
-- Statewide Turnout Summary by Year
SELECT 
    YEAR,
    COUNT(DISTINCT COUNTY) AS COUNTIES,
    COUNT(DISTINCT PRECINCT) AS PRECINCTS,
    SUM(TOTAL_VOTES) AS TOTAL_VOTES,
    ROUND(SUM(REPUBLICAN_VOTES) / SUM(TOTAL_VOTES) * 100, 2) AS STATEWIDE_REP_PCT,
    ROUND(SUM(DEMOCRAT_VOTES) / SUM(TOTAL_VOTES) * 100, 2) AS STATEWIDE_DEM_PCT
FROM PRECINCT_TURNOUT
GROUP BY YEAR
ORDER BY YEAR;
"""
df = pd.read_sql(sql, conn)
df

,YEAR,COUNTIES,PRECINCTS,TOTAL_VOTES,STATEWIDE_REP_PCT,STATEWIDE_DEM_PCT
0,2016,116,3048,2808298.0,56.78,38.14
1,2020,116,3480,2963270.0,57.17,41.03
2,2024,116,3063,2995327.0,58.49,40.08


In [33]:
# Execute SQL Query
sql = """
-- County Turnout Descriptive Statistics (2020)
SELECT 
    '2020 County Stats' AS METRIC,
    COUNT(*) AS N_COUNTIES,
    ROUND(AVG(TOTAL_VOTES), 0) AS MEAN_VOTES,
    ROUND(MEDIAN(TOTAL_VOTES), 0) AS MEDIAN_VOTES,
    MIN(TOTAL_VOTES) AS MIN_VOTES,
    MAX(TOTAL_VOTES) AS MAX_VOTES,
    ROUND(STDDEV(TOTAL_VOTES), 0) AS STD_VOTES
FROM COUNTY_TURNOUT
WHERE YEAR = 2020;
"""
df = pd.read_sql(sql, conn)
df

,METRIC,N_COUNTIES,MEAN_VOTES,MEDIAN_VOTES,MIN_VOTES,MAX_VOTES,STD_VOTES
0,2020 County Stats,116,25545.0,9119.0,1107.0,535998.0,59431.0


In [34]:
# Execute SQL Query
sql = """
-- Top 10 Counties by Total Votes (2020)
SELECT 
    COUNTY,
    TOTAL_VOTES,
    REPUBLICAN_PCT,
    DEMOCRAT_PCT,
    PRECINCT_COUNT
FROM COUNTY_TURNOUT
WHERE YEAR = 2020
ORDER BY TOTAL_VOTES DESC
LIMIT 10;
"""
df = pd.read_sql(sql, conn)
df

,COUNTY,TOTAL_VOTES,REPUBLICAN_PCT,DEMOCRAT_PCT,PRECINCT_COUNT
0,ST. LOUIS COUNTY,535998.0,37.18,61.22,1235
1,ST. CHARLES,222017.0,57.83,40.33,124
2,GREENE,141800.0,58.98,38.83,80
3,KANSAS CITY,136645.0,19.32,78.79,106
4,JACKSON,136300.0,57.17,40.32,148
5,ST. LOUIS CITY,133867.0,16.04,82.24,223
6,CLAY,126569.0,51.04,46.93,82
7,JEFFERSON,116688.0,66.03,32.16,55
8,BOONE,91168.0,42.39,54.91,83
9,CASS,57408.0,64.79,33.19,42


In [35]:
# Execute SQL Query
sql = """
-- Top 10 Counties by Turnout Rate (2020)
SELECT 
    COUNTY,
    TURNOUT_PCT_2020,
    VOTES_2020,
    VAP_2020 AS VOTING_AGE_POP,
    POP_2020 AS TOTAL_POP
FROM COUNTY_ANALYSIS
WHERE TURNOUT_PCT_2020 IS NOT NULL
ORDER BY TURNOUT_PCT_2020 DESC
LIMIT 10;
"""
df = pd.read_sql(sql, conn)
df

,COUNTY,TURNOUT_PCT_2020,VOTES_2020,VOTING_AGE_POP,TOTAL_POP
0,SHELBY,73.74,3350.0,4543,5975
1,PLATTE,73.01,57270.0,78444,102848
2,ST. CHARLES,72.65,222017.0,305584,398472
3,CASS,72.24,57408.0,79463,104687
4,ANDREW,72.13,9752.0,13520,17554
5,OSAGE,71.90,7539.0,10486,13613
6,CHRISTIAN,71.88,46839.0,65161,87324
7,CHARITON,70.83,4077.0,5756,7449
8,RALLS,70.29,5662.0,8055,10258
9,WORTH,70.02,1107.0,1581,2001


In [36]:
# Execute SQL Query
sql = """
-- Bottom 10 Counties by Turnout Rate (2020)
SELECT 
    COUNTY,
    TURNOUT_PCT_2020,
    VOTES_2020,
    VAP_2020 AS VOTING_AGE_POP,
    POP_2020 AS TOTAL_POP
FROM COUNTY_ANALYSIS
WHERE TURNOUT_PCT_2020 IS NOT NULL
ORDER BY TURNOUT_PCT_2020 ASC
LIMIT 10;
"""
df = pd.read_sql(sql, conn)
df

,COUNTY,TURNOUT_PCT_2020,VOTES_2020,VOTING_AGE_POP,TOTAL_POP
0,JACKSON,25.45,136300.0,535522,700733
1,PULASKI,35.25,14414.0,40886,52359
2,MISSISSIPPI,45.14,4756.0,10536,13328
3,DUNKLIN,47.24,10419.0,22057,29657
4,PEMISCOT,47.32,5735.0,12119,16330
5,ADAIR,49.93,10336.0,20700,25468
6,SULLIVAN,51.73,2471.0,4777,6163
7,STE. GENEVIEVE,52.33,7358.0,14061,17887
8,WASHINGTON,52.39,9976.0,19043,24819
9,NODAWAY,52.98,9893.0,18674,22199


In [37]:
# Execute SQL Query
sql = """
-- Turnout Change 2016 to 2024
SELECT 
    COUNTY,
    VOTES_2016,
    VOTES_2020,
    VOTES_2024,
    VOTES_2024 - VOTES_2016 AS VOTE_CHANGE,
    ROUND((VOTES_2024 - VOTES_2016) / NULLIF(VOTES_2016, 0) * 100, 2) AS PCT_CHANGE
FROM COUNTY_TURNOUT_TREND
WHERE VOTES_2016 IS NOT NULL AND VOTES_2024 IS NOT NULL
ORDER BY PCT_CHANGE DESC
LIMIT 10;
"""
df = pd.read_sql(sql, conn)
df

,COUNTY,VOTES_2016,VOTES_2020,VOTES_2024,VOTE_CHANGE,PCT_CHANGE
0,LINCOLN,24962.0,29018.0,32133.0,7171.0,28.73
1,WARREN,15784.0,18375.0,20095.0,4311.0,27.31
2,CHRISTIAN,41531.0,46839.0,50789.0,9258.0,22.29
3,WEBSTER,16743.0,18779.0,19814.0,3071.0,18.34
4,CLAY,110176.0,126569.0,130160.0,19984.0,18.14
5,PLATTE,49100.0,57270.0,57817.0,8717.0,17.75
6,CASS,50990.0,57408.0,59349.0,8359.0,16.39
7,ST. FRANCOIS,24728.0,28001.0,28615.0,3887.0,15.72
8,STE. GENEVIEVE,8469.0,7358.0,9776.0,1307.0,15.43
9,STONE,16567.0,18483.0,19123.0,2556.0,15.43


In [38]:
# Execute SQL Query
sql = """
-- Polling Resource Strain Analysis
SELECT 
    p.COUNTY,
    p.UNIQUE_POLLING_PLACES,
    p.UNIQUE_PRECINCTS,
    p.PRECINCTS_PER_POLLING_PLACE,
    a.POP_2020,
    a.VOTES_2020,
    a.TURNOUT_PCT_2020
FROM COUNTY_POLLING_SUMMARY p
LEFT JOIN COUNTY_ANALYSIS a ON p.COUNTY = a.COUNTY
WHERE p.PRECINCTS_PER_POLLING_PLACE > 1
ORDER BY p.PRECINCTS_PER_POLLING_PLACE DESC
LIMIT 15;
"""
df = pd.read_sql(sql, conn)
df

,COUNTY,UNIQUE_POLLING_PLACES,UNIQUE_PRECINCTS,PRECINCTS_PER_POLLING_PLACE,POP_2020,VOTES_2020,TURNOUT_PCT_2020
0,SULLIVAN,3,67,22.33,6163,2471.0,51.73
1,DAVIESS,5,110,22.00,8294,3911.0,63.46
2,RIPLEY,6,118,19.67,13484,5717.0,54.81
3,SALINE,11,209,19.00,22932,9519.0,53.83
4,CLINTON,6,97,16.17,20503,10888.0,69.44
5,PULASKI,8,114,14.25,52359,14414.0,35.25
6,MONROE,7,95,13.57,8630,4483.0,66.31
7,HOLT,4,54,13.50,4374,2343.0,67.23
8,MORGAN,10,134,13.40,20438,9487.0,60.17
9,JOHNSON,10,127,12.70,53948,23123.0,54.59


In [39]:
# Execute SQL Query
sql = """
-- Demographics vs Turnout Snapshot
SELECT 
    COUNTY,
    POP_2020,
    INCOME_2020,
    EDU_2020 AS PCT_BACHELORS,
    MINORITY_2020 AS PCT_MINORITY,
    NO_VEHICLE_2020 AS PCT_NO_VEHICLE,
    TURNOUT_PCT_2020,
    REP_PCT_2020,
    DEM_PCT_2020
FROM COUNTY_ANALYSIS
WHERE TURNOUT_PCT_2020 IS NOT NULL
ORDER BY TURNOUT_PCT_2020 DESC
LIMIT 20;
"""
df = pd.read_sql(sql, conn)
df

,COUNTY,POP_2020,INCOME_2020,PCT_BACHELORS,PCT_MINORITY,PCT_NO_VEHICLE,TURNOUT_PCT_2020,REP_PCT_2020,DEM_PCT_2020
0,SHELBY,5975,43809,19.38,4.89,8.89,73.74,80.60,17.67
1,PLATTE,102848,82448,43.00,16.04,0.84,73.01,50.49,47.46
2,ST. CHARLES,398472,87644,40.76,11.21,0.84,72.65,57.83,40.33
3,CASS,104687,72522,27.00,9.83,1.12,72.24,64.79,33.19
4,ANDREW,17554,58911,26.56,4.93,2.00,72.13,74.39,24.11
5,OSAGE,13613,62087,20.44,1.88,3.48,71.90,85.22,13.76
6,CHRISTIAN,87324,64442,30.67,5.66,1.57,71.88,74.55,23.76
7,CHARITON,7449,51545,18.46,5.20,3.61,70.83,76.31,22.47
8,RALLS,10258,54194,15.37,5.52,1.69,70.29,77.64,21.28
9,WORTH,2001,47500,18.28,1.70,6.46,70.02,79.22,19.42


In [40]:
# Correlation Analysis - Demographics vs Turnout

county_analysis_df = pd.read_sql('SELECT * FROM COUNTY_ANALYSIS', conn)

print("=" * 70)
print("CORRELATION MATRIX - DEMOGRAPHICS VS TURNOUT (2020)")
print("=" * 70)

correlation_cols = [
    'INCOME_2020', 'EDU_2020', 'MINORITY_2020', 'NO_VEHICLE_2020',
    'TURNOUT_PCT_2020', 'REP_PCT_2020', 'DEM_PCT_2020'
]

valid_cols = [c for c in correlation_cols if c in county_analysis_df.columns]
correlation_matrix = county_analysis_df[valid_cols].corr().round(3)
print(correlation_matrix)
print("=" * 70)

CORRELATION MATRIX - DEMOGRAPHICS VS TURNOUT (2020)
                  INCOME_2020  EDU_2020  MINORITY_2020  NO_VEHICLE_2020  \
INCOME_2020             1.000     0.670          0.121           -0.080   
EDU_2020                0.670     1.000          0.488            0.220   
MINORITY_2020           0.121     0.488          1.000            0.365   
NO_VEHICLE_2020        -0.080     0.220          0.365            1.000   
TURNOUT_PCT_2020        0.343     0.078         -0.539           -0.260   
REP_PCT_2020           -0.471    -0.757         -0.701           -0.251   
DEM_PCT_2020            0.459     0.747          0.707            0.248   

                  TURNOUT_PCT_2020  REP_PCT_2020  DEM_PCT_2020  
INCOME_2020                  0.343        -0.471         0.459  
EDU_2020                     0.078        -0.757         0.747  
MINORITY_2020               -0.539        -0.701         0.707  
NO_VEHICLE_2020             -0.260        -0.251         0.248  
TURNOUT_PCT_2020      

In [41]:
# Key Findings Summary

print("=" * 70)
print("KEY FINDINGS SUMMARY")
print("=" * 70)

print("""
1. STATEWIDE TRENDS:
   - Compare total turnout across 2016, 2020, 2024
   - Analyze party vote share changes over time
   
2. TURNOUT PATTERNS:
   - Identify high and low turnout counties
   - Examine turnout rate vs raw vote counts
   - Track changes in turnout over time

3. DEMOGRAPHIC CORRELATIONS:
   - Income vs turnout relationship
   - Education level impact on participation
   - Minority population voting patterns
   - Transportation access and voting

4. POLLING RESOURCES:
   - Counties with high precincts-per-polling-place
   - Cross-reference with population and turnout
   - Identify potential under-resourced areas

5. NEXT STEPS:
   - Predictive modeling for voter demand
   - Prescriptive recommendations for resource allocation
   - Geospatial visualization (David's section)
   - Final presentation preparation
""")
print("=" * 70)

KEY FINDINGS SUMMARY

1. STATEWIDE TRENDS:
   - Compare total turnout across 2016, 2020, 2024
   - Analyze party vote share changes over time
   
2. TURNOUT PATTERNS:
   - Identify high and low turnout counties
   - Examine turnout rate vs raw vote counts
   - Track changes in turnout over time

3. DEMOGRAPHIC CORRELATIONS:
   - Income vs turnout relationship
   - Education level impact on participation
   - Minority population voting patterns
   - Transportation access and voting

4. POLLING RESOURCES:
   - Counties with high precincts-per-polling-place
   - Cross-reference with population and turnout
   - Identify potential under-resourced areas

5. NEXT STEPS:
   - Predictive modeling for voter demand
   - Prescriptive recommendations for resource allocation
   - Geospatial visualization (David's section)
   - Final presentation preparation



---
# SECTION 10: Geospatial Analysis
---

**Owner: David**

This section is reserved for geospatial analysis using the precinct boundary shapefiles. Potential analyses include:
- Mapping voter turnout by precinct
- Visualizing polling location distribution
- Calculating distances between population centers and polling places
- Identifying geographic clusters of under-resourced areas

In [42]:
# Geospatial Analysis - David (STUB)

# =============================================================================
# GEOSPATIAL ANALYSIS - DAVID
# =============================================================================
#
# This cell is a starting point for geospatial analysis.
# The shapefile data has been loaded and the tabular version is in STG_PRECINCTS.
#
# AVAILABLE DATA:
# ---------------
# - raw_precincts_gdf: GeoDataFrame with full geometry (loaded in Cell 12)
# - STG_PRECINCTS: Tabular precinct data with centroids (lat/lon)
#
# POTENTIAL ANALYSES:
# -------------------
# 1. Choropleth map of turnout by precinct
# 2. Polling location accessibility analysis
# 3. Distance calculations (voters to nearest polling place)
# 4. Geographic clustering of low-turnout areas
#
# SUGGESTED LIBRARIES:
# --------------------
# - geopandas (already imported)
# - folium (for interactive maps)
# - shapely (for geometry operations)
#
# =============================================================================

# David - add your geospatial analysis code below:

# print("=" * 70)
# print("GEOSPATIAL ANALYSIS - DAVID")
# print("=" * 70)
# print("Shapefile loaded: MO 2020 Precincts")
# print(f"Total precincts: {raw_precincts_gdf.shape[0]:,}")
# print(f"CRS: {raw_precincts_gdf.crs}")
# print(f"Geometry types: {raw_precincts_gdf.geom_type.unique().tolist()}")
print("=" * 70)
# 
# Example: View first few precincts
# raw_precincts_gdf[['NAME20', 'COUNTYFP20', 'ALAND20', 'INTPTLAT20', 'INTPTLON20']].head()

In [43]:
# Execute SQL: Create/Update Table
sql = """
-- Create Visualization Export Table
CREATE OR REPLACE TABLE COUNTY_VIZ_EXPORT AS
SELECT 
    a.COUNTY,
    a.POP_2016, a.POP_2020, a.POP_2024,
    a.VAP_2016, a.VAP_2020, a.VAP_2024,
    a.VOTES_2016, a.VOTES_2020, a.VOTES_2024,
    a.TURNOUT_PCT_2016, a.TURNOUT_PCT_2020, a.TURNOUT_PCT_2024,
    a.REP_PCT_2016, a.REP_PCT_2020, a.REP_PCT_2024,
    a.DEM_PCT_2016, a.DEM_PCT_2020, a.DEM_PCT_2024,
    a.INCOME_2016, a.INCOME_2020, a.INCOME_2024,
    a.EDU_2016, a.EDU_2020, a.EDU_2024,
    a.MINORITY_2016, a.MINORITY_2020, a.MINORITY_2024,
    a.NO_VEHICLE_2020,
    p.UNIQUE_POLLING_PLACES,
    p.UNIQUE_PRECINCTS,
    p.PRECINCTS_PER_POLLING_PLACE
FROM COUNTY_ANALYSIS a
LEFT JOIN COUNTY_POLLING_SUMMARY p ON a.COUNTY = p.COUNTY
ORDER BY a.COUNTY;
"""
cursor.execute(sql)
print("Table created/updated successfully")

Table created/updated successfully


---
# SECTION 11: Summary & Next Steps
---

## Data Pipeline Complete

### Staging Tables Created:
- `STG_ELECTION_RESULTS` - Combined presidential results (2016, 2020, 2024)
- `STG_CENSUS_INCOME` - Median household income (3 years)
- `STG_CENSUS_EDUCATION` - Educational attainment (3 years)
- `STG_CENSUS_RACE` - Race demographics (3 years)
- `STG_CENSUS_COMMUTE` - Transportation/commute patterns (3 years)
- `STG_CENSUS_SEX_AGE` - Age/sex distribution with VAP (3 years)
- `STG_POLLING_LOCATIONS` - Polling place locations (2020)
- `STG_PRECINCTS` - Precinct boundary data (2020)

### Analytical Tables Created:
- `PRECINCT_TURNOUT` - Precinct-level turnout by year
- `COUNTY_TURNOUT` - County-level turnout by year
- `COUNTY_TURNOUT_TREND` - Turnout pivoted across years
- `COUNTY_CENSUS` - Census demographics by year
- `COUNTY_ANALYSIS` - Master analysis table with all metrics
- `COUNTY_POLLING_SUMMARY` - Polling resource distribution
- `COUNTY_VIZ_EXPORT` - Export-ready for visualization tools

## Next Steps for Team:
1. **Predictive Modeling** - Build models to predict voter demand
2. **Prescriptive Analytics** - Recommend polling resource allocation
3. **Geospatial Analysis** - David's precinct mapping work
4. **Visualization** - Tableau/Flourish dashboards
5. **Final Presentation** - Synthesize findings

In [44]:
# Final Status

print("=" * 70)
print("NOTEBOOK EXECUTION COMPLETE")
print("=" * 70)
print("""
DATA LOADED:
  • 3 Election years (2016, 2020, 2024) - Presidential results only
  • 15 Census files (5 tables × 3 years)
  • 1 Polling locations file (2020)
  • 1 Shapefile set (precinct boundaries)

TABLES CREATED: 14 total (8 staging + 6 analytical)

READY FOR:
  • Predictive modeling
  • Prescriptive analytics
  • Geospatial analysis (David)
  • Visualization exports
  
COLLABORATION NOTE:
  This notebook will be ported to a shared team GitHub account
  for full team collaboration.
""")
print("=" * 70)

NOTEBOOK EXECUTION COMPLETE

DATA LOADED:
  • 3 Election years (2016, 2020, 2024) - Presidential results only
  • 15 Census files (5 tables × 3 years)
  • 1 Polling locations file (2020)
  • 1 Shapefile set (precinct boundaries)

TABLES CREATED: 14 total (8 staging + 6 analytical)

READY FOR:
  • Predictive modeling
  • Prescriptive analytics
  • Geospatial analysis (David)
  • Visualization exports
  
COLLABORATION NOTE:
  This notebook will be ported to a shared team GitHub account
  for full team collaboration.



---
# SECTION XX: Various Analyses
---

## Data Analyses (for temporary awareness in building our deliverables)



In [45]:
# Execute SQL Query
sql = """
SELECT 
    a.COUNTY,
    a.VOTES_2020,
    p.UNIQUE_POLLING_PLACES,
    ROUND(a.VOTES_2020 / NULLIF(p.UNIQUE_POLLING_PLACES, 0), 0) AS VOTERS_PER_POLLING_PLACE,
    p.PRECINCTS_PER_POLLING_PLACE,
    a.TURNOUT_PCT_2020
FROM COUNTY_ANALYSIS a
LEFT JOIN COUNTY_POLLING_SUMMARY p ON a.COUNTY = p.COUNTY
WHERE a.VOTES_2020 IS NOT NULL
ORDER BY VOTERS_PER_POLLING_PLACE DESC
LIMIT 15;
"""
df = pd.read_sql(sql, conn)
df

,COUNTY,VOTES_2020,UNIQUE_POLLING_PLACES,VOTERS_PER_POLLING_PLACE,PRECINCTS_PER_POLLING_PLACE,TURNOUT_PCT_2020
0,ST. LOUIS COUNTY,535998.0,NaN,NaN,NaN,NaN
1,JOHNSON,23123.0,10.0,2312.0,12.70,54.59
2,JEFFERSON,116688.0,53.0,2202.0,7.66,67.57
3,PLATTE,57270.0,27.0,2121.0,8.85,73.01
4,MADISON,5676.0,3.0,1892.0,7.33,60.80
5,CHRISTIAN,46839.0,25.0,1874.0,7.32,71.88
6,GREENE,141800.0,76.0,1866.0,3.25,61.40
7,ST. CHARLES,222017.0,122.0,1820.0,5.56,72.65
8,CLINTON,10888.0,6.0,1815.0,16.17,69.44
9,PULASKI,14414.0,8.0,1802.0,14.25,35.25


In [46]:
# Execute SQL Query
sql = """
SELECT 
    COUNTY,
    INCOME_2020,
    TURNOUT_PCT_2020,
    NO_VEHICLE_2020
FROM COUNTY_ANALYSIS
WHERE TURNOUT_PCT_2020 IS NOT NULL
ORDER BY INCOME_2020 DESC;
"""
df = pd.read_sql(sql, conn)
df

,COUNTY,INCOME_2020,TURNOUT_PCT_2020,NO_VEHICLE_2020
0,ST. CHARLES,87644,72.65,0.84
1,PLATTE,82448,73.01,0.84
2,CASS,72522,72.24,1.12
3,CLAY,72047,67.67,1.55
4,LINCOLN,70424,67.58,2.38
...,...,...,...,...
108,PEMISCOT,34709,47.32,1.81
109,MISSISSIPPI,34354,45.14,3.71
110,HICKORY,33342,64.49,0.67
111,OZARK,33046,65.78,6.09


In [47]:
# Execute SQL Query
sql = """
SELECT 
    ROUND(AVG(TURNOUT_PCT_2020), 2) AS AVG_TURNOUT,
    ROUND(MEDIAN(TURNOUT_PCT_2020), 2) AS MEDIAN_TURNOUT,
    MIN(TURNOUT_PCT_2020) AS MIN_TURNOUT,
    MAX(TURNOUT_PCT_2020) AS MAX_TURNOUT
FROM COUNTY_ANALYSIS
WHERE TURNOUT_PCT_2020 IS NOT NULL;
"""
df = pd.read_sql(sql, conn)
df

,AVG_TURNOUT,MEDIAN_TURNOUT,MIN_TURNOUT,MAX_TURNOUT
0,61.59,62.92,25.45,73.74


In [48]:
# Execute SQL Query
sql = """
SELECT DISTINCT "county_clean" 
FROM STG_POLLING_LOCATIONS 
WHERE "county_clean" LIKE '%ST. LOUIS%' OR "county_clean" LIKE '%SAINT LOUIS%';
"""
df = pd.read_sql(sql, conn)
df

,county_clean
0,ST. LOUIS
1,ST. LOUIS CITY


In [49]:
# Execute SQL Query
sql = """
SELECT "county_clean", COUNT(*) as polling_records
FROM STG_POLLING_LOCATIONS
GROUP BY "county_clean"
ORDER BY "county_clean";
"""
df = pd.read_sql(sql, conn)
df

,county_clean,POLLING_RECORDS
0,ADAIR,49
1,ANDREW,80
2,ATCHISON,55
3,AUDRAIN,127
4,BARRY,169
...,...,...
111,WASHINGTON,70
112,WAYNE,53
113,WEBSTER,64
114,WORTH,30


In [50]:
# Execute SQL Query
sql = """
SELECT COUNTY FROM COUNTY_ANALYSIS WHERE COUNTY LIKE '%ST. LOUIS%';
"""
df = pd.read_sql(sql, conn)
df

,COUNTY
0,ST. LOUIS
1,ST. LOUIS CITY
2,ST. LOUIS COUNTY


In [51]:
# Execute SQL Query
sql = """
SELECT COUNTY FROM COUNTY_POLLING_SUMMARY WHERE COUNTY LIKE '%ST. LOUIS%';
"""
df = pd.read_sql(sql, conn)
df

,COUNTY
0,ST. LOUIS
1,ST. LOUIS CITY


In [52]:
# Execute SQL Query
sql = """
SELECT COUNTY, VOTES_2020 
FROM COUNTY_ANALYSIS 
WHERE COUNTY LIKE '%ST. LOUIS%';
"""
df = pd.read_sql(sql, conn)
df

,COUNTY,VOTES_2020
0,ST. LOUIS,NaN
1,ST. LOUIS CITY,133867.0
2,ST. LOUIS COUNTY,535998.0


In [53]:
# Execute SQL Query
sql = """
SELECT 
    a.COUNTY,
    a.VOTES_2020,
    p.UNIQUE_POLLING_PLACES,
    ROUND(a.VOTES_2020 / NULLIF(p.UNIQUE_POLLING_PLACES, 0), 0) AS VOTERS_PER_POLLING_PLACE
FROM COUNTY_ANALYSIS a
LEFT JOIN COUNTY_POLLING_SUMMARY p 
    ON a.COUNTY = p.COUNTY 
    OR (a.COUNTY = 'ST. LOUIS COUNTY' AND p.COUNTY = 'ST. LOUIS')
WHERE a.COUNTY LIKE '%ST. LOUIS%';
"""
df = pd.read_sql(sql, conn)
df

,COUNTY,VOTES_2020,UNIQUE_POLLING_PLACES,VOTERS_PER_POLLING_PLACE
0,ST. LOUIS,NaN,230,NaN
1,ST. LOUIS COUNTY,535998.0,230,2330.0
2,ST. LOUIS CITY,133867.0,253,529.0


In [54]:
import geopandas as gpd
print(gpd.__version__)

1.1.3


In [55]:
# Execute SQL Query
sql = """
SELECT * FROM STG_POLLING_LOCATIONS LIMIT 5;
"""
df = pd.read_sql(sql, conn)
df

,election_date,state,county_name,jurisdiction,jurisdiction_type,precinct_id,precinct_name,polling_place_id,location_type,name,address,notes,source,source_date,source_notes,county_clean,precinct_clean,polling_address
0,2020-11-03,MO,Adair,Adair,county,1009-03,BRASHEAR/WILSON 03,BRASHEAR1,None,NEMO FAIR GROUNDS,"2700 E ILLINOIS ST KIRKSVILLE, 63501",None,website,2020-10-06,None,ADAIR,BRASHEAR/WILSON 03,"2700 E ILLINOIS ST KIRKSVILLE, 63501"
1,2020-11-03,MO,Adair,Adair,county,1009-04,BRASHEAR/WILSON 04,BRASHEAR1,None,NEMO FAIR GROUNDS,"2700 E ILLINOIS ST KIRKSVILLE, 63501",None,website,2020-10-06,None,ADAIR,BRASHEAR/WILSON 04,"2700 E ILLINOIS ST KIRKSVILLE, 63501"
2,2020-11-03,MO,Adair,Adair,county,1009-05,BRASHEAR/WILSON 05,BRASHEAR1,None,NEMO FAIR GROUNDS,"2700 E ILLINOIS ST KIRKSVILLE, 63501",None,website,2020-10-06,None,ADAIR,BRASHEAR/WILSON 05,"2700 E ILLINOIS ST KIRKSVILLE, 63501"
3,2020-11-03,MO,Adair,Adair,county,1009-07,BRASHEAR/WILSON 07,BRASHEAR1,None,NEMO FAIR GROUNDS,"2700 E ILLINOIS ST KIRKSVILLE, 63501",None,website,2020-10-06,None,ADAIR,BRASHEAR/WILSON 07,"2700 E ILLINOIS ST KIRKSVILLE, 63501"
4,2020-11-03,MO,Adair,Adair,county,1009-08,BRASHEAR/WILSON 08,BRASHEAR1,None,NEMO FAIR GROUNDS,"2700 E ILLINOIS ST KIRKSVILLE, 63501",None,website,2020-10-06,None,ADAIR,BRASHEAR/WILSON 08,"2700 E ILLINOIS ST KIRKSVILLE, 63501"
